# Stage III. LG-DRG: five-fold training

This notebook trains the Lesion-Guided DR Grader (LG-DRG) on the 18,570 DR-positive
development images (Grades 1-5), one model per stratified fold.

**Model.** An RGB stream (ConvNeXt V2-Large, ImageNet-22k/1k) and a light mask stream (three
stride-2 conv-BN-GELU blocks over the five Stage II masks) meet in a residual cross-attention
gate, `F_gated = F_img + gamma * W_o A`, where mask tokens query image tokens (8 heads, width
256) and `gamma` starts at 0. After BN and global average pooling, a gradability head predicts
`P(gradable)` (Grade 5 = ungradable, refer) and a CORN head models severity Grades 1-4.

**Steps.**
1. Build the development table, assign five stratified folds and verify every file.
2. Compute the gradability loss weight and inspect the model.
3. Train the five folds (resumable per epoch).
4. Summarise the best validation macro-AUC per fold.

## 0. Configuration

In [ ]:
from pathlib import Path

# Development pool (18,570 DR-positive images, grades 1-5) and its Stage II masks.
#   data/stage3/dev_grades.csv              columns: image, grade (1..5)
#   data/stage3/dev_images/<stem>.png|jpg
#   data/stage3/dev_masks/<lesion>/<stem><suffix>.png   lesion in {ma, he, ex, cws, vessel};
#       suffix: ma "_mask", he "_he_mask", ex "_ex_mask", cws "_cws_mask", vessel "_mask"
DATA_ROOT = Path("data")
DEV_CSV = DATA_ROOT / "stage3" / "dev_grades.csv"
DEV_IMG_DIR = DATA_ROOT / "stage3" / "dev_images"
DEV_MASK_ROOT = DATA_ROOT / "stage3" / "dev_masks"

OUT_ROOT = Path("outputs")
CKPT_DIR = OUT_ROOT / "stage3" / "checkpoints"      # best_fold{0..4}.pt
EVAL_DIR = OUT_ROOT / "stage3" / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

import torch
from hierarchiretina.stage3.engine import LGDRGConfig

# Hyper-parameters as stored in the released checkpoints: batch 16 with gradient
# accumulation 4 (effective batch 64). The fold-0 gamma-fix retrain used accumulation 8.
cfg = LGDRGConfig(out_dir=str(CKPT_DIR))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
cfg

## 1. Development table and folds

Each image is resolved to its file and its five mask files. Folds come from
`StratifiedKFold(5, shuffle=True, random_state=42)` on the grade, applied in CSV order, so the
CSV row order must not change. Missing masks would be zero-filled at load time and silently
degrade training, hence the check.

In [ ]:
from hierarchiretina.stage3.data import prepare_development_set, grad_pos_weight

df = prepare_development_set(DEV_CSV, DEV_IMG_DIR, DEV_MASK_ROOT, cfg.n_folds, cfg.seed)
print(df["grade"].value_counts().sort_index())

## 2. Loss weight and model

The gradability BCE treats *gradable* as the positive class; its `pos_weight` is
n_ungradable / n_gradable over the development pool (805 / 17,765 = 0.045), so the rare
ungradable images are not swamped. The CORN loss uses gradable images only.

In [ ]:
from hierarchiretina.stage3.model import LGDRG

POS_WEIGHT = grad_pos_weight(df)
print(f"gradability pos_weight = {POS_WEIGHT:.4f}")

with torch.device("meta"):                     # parameter count without allocating memory
    _m = LGDRG(cfg, pretrained=False)
print(f"LG-DRG parameters: {sum(p.numel() for p in _m.parameters()) / 1e6:.1f} M")

## 3. Train the five folds

Per fold: AdamW (heads 2e-4, backbone 2e-5, weight decay 0.05 on all parameters), linear
warm-up then cosine (sized in batches but stepped per accumulated update, as in the original
run), AMP, clip-norm 5, EMA 0.9998, inverse-frequency sampling over the five grades, batch 16
with gradient accumulation, at most 60 epochs, patience 12, selection on the EMA model's
five-class macro-AUC. Each fold writes `best_fold{k}.pt`, `last_fold{k}.pt`,
`history_fold{k}.json` and a `done_fold{k}.flag`; re-running resumes or skips.

In [ ]:
from hierarchiretina.stage3.engine import train_all_folds

summary = train_all_folds(cfg, df, POS_WEIGHT, device=DEVICE)

## 4. Summary

In [ ]:
print(summary.to_string(index=False))
print(f"mean +/- sd: {summary.best_macro_auc.mean():.4f} +/- {summary.best_macro_auc.std(ddof=0):.4f}")